# Named Entity Recognition (NER) with PyTorch and Transformers

This notebook is from [AI for Beginners Curriculum](http://aka.ms/ai-beginners).

In the [previous notebook](NER-TF.ipynb) we trained a NER model from scratch. In this notebook we will take a different approach that is much closer to what is used in practice today: we will **fine-tune a small pre-trained transformer** ([DistilBERT](https://huggingface.co/distilbert/distilbert-base-cased)) to do token classification.

We will use a small part of the [CoNLL-2003](https://huggingface.co/datasets/tomaarsen/conll2003) dataset, which contains news sentences labeled with four entity types: persons (`PER`), organizations (`ORG`), locations (`LOC`) and miscellaneous names (`MISC`). The notebook is designed to run on a regular laptop CPU in a few minutes, and it will use a GPU automatically if you have one.

Here is what we are going to do:

1. Load the dataset and look at the BIO tags
2. Tokenize the text and align word-level labels with sub-word tokens
3. Fine-tune the model with a simple PyTorch training loop
4. Evaluate it using entity-level precision, recall and F1 (and see why token accuracy is misleading)
5. Try the model on our own sentences

In [ ]:
# The packages below are used in this notebook. If some of them are missing in your environment, uncomment and run:
# %pip install transformers datasets seqeval

import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, get_linear_schedule_with_warmup
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# Make the experiment reproducible
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Use a GPU if there is one, otherwise the CPU is fine for this notebook
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Loading the Dataset

We will load CoNLL-2003 directly from the Hugging Face Hub using the `datasets` library. The full training set contains around 14,000 sentences. To keep training fast, we will only use a **random subset**: 2,000 sentences for training, and 500 each for validation and testing. Feel free to increase these numbers if you have more time (or a GPU).

In [ ]:
raw = load_dataset('tomaarsen/conll2003')

train_data = raw['train'].shuffle(seed=seed).select(range(2000))
val_data = raw['validation'].shuffle(seed=seed).select(range(500))
test_data = raw['test'].shuffle(seed=seed).select(range(500))

train_data

Each example contains a list of words (`tokens`) and a list of tag numbers (`ner_tags`), one for each word. The names of the tags are stored in the dataset features:

In [ ]:
label_names = raw['train'].features['ner_tags'].feature.names
id2label = dict(enumerate(label_names))
label2id = {v: k for k, v in id2label.items()}
print(label_names)

We have 4 entity types, so we have 2*4+1 = 9 classes: a `B-` and an `I-` tag for each type, and `O` for all other words. This is exactly the BIO tagging scheme described in the lesson. Let's print a few training sentences with their tags:

In [ ]:
for example in train_data.select(range(3)):
    print(' '.join(f'{word}/{id2label[tag]}' for word, tag in zip(example['tokens'], example['ner_tags'])))
    print()

Let's also check how many of the words are tagged as `O`. This is important for the evaluation later:

In [ ]:
all_tags = [tag for tags in train_data['ner_tags'] for tag in tags]
print(f"Words that are not part of any entity: {np.mean(np.array(all_tags) == label2id['O']):.1%}")

## Tokenization and Label Alignment

Transformer models do not work with words, but with **sub-word tokens**. We need to use the tokenizer that belongs to the model we are going to fine-tune. We will use `distilbert-base-cased`: a smaller and faster version of BERT that keeps most of its quality. The *cased* version is a good choice for NER, because capital letters are a strong hint that a word is a name.

In [ ]:
model_name = 'distilbert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

Our sentences are already split into words, so we tell the tokenizer about this with `is_split_into_words=True`. Let's see what happens to a sentence with some rare words:

In [ ]:
words = ['Sundar', 'Pichai', 'visited', 'Hyderabad', '.']
encoding = tokenizer(words, is_split_into_words=True)

print(tokenizer.convert_ids_to_tokens(encoding['input_ids']))
print(encoding.word_ids())

The first line shows that some words were split into several pieces (the `##` prefix marks a continuation of a word), and that the special tokens `[CLS]` and `[SEP]` were added at the beginning and at the end. The second line is the output of `word_ids()`: it tells us which word each token belongs to (`None` for special tokens).

This means that we have more tokens than word-level labels, so we cannot use the labels as they are. We will use the following strategy:

* The **first** sub-word token of each word gets the label of the word.
* All other tokens (the special tokens and the continuation pieces) get the label `-100`. This is a special value that is ignored by the PyTorch loss function, so these positions will not influence training.

Let's write a function that tokenizes a batch of sentences and aligns the labels:

In [ ]:
def tokenize_and_align(batch):
    encoding = tokenizer(batch['tokens'], is_split_into_words=True, truncation=True, max_length=128)
    all_labels = []
    for i, word_tags in enumerate(batch['ner_tags']):
        word_ids = encoding.word_ids(batch_index=i)
        labels = []
        previous_word = None
        for word_id in word_ids:
            if word_id is None:                # special token
                labels.append(-100)
            elif word_id != previous_word:     # first sub-word token of a word
                labels.append(word_tags[word_id])
            else:                              # continuation of the same word
                labels.append(-100)
            previous_word = word_id
        all_labels.append(labels)
    encoding['labels'] = all_labels
    return encoding

columns = train_data.column_names
train_tok = train_data.map(tokenize_and_align, batched=True, remove_columns=columns)
val_tok = val_data.map(tokenize_and_align, batched=True, remove_columns=columns)
test_tok = test_data.map(tokenize_and_align, batched=True, remove_columns=columns)

Let's look at one aligned example to make sure that everything is correct:

In [ ]:
example = train_tok[0]
tokens = tokenizer.convert_ids_to_tokens(example['input_ids'])
for token, label in zip(tokens, example['labels']):
    print(f"{token:15} {id2label[label] if label != -100 else '-100'}")

Sentences have different lengths, so to process them in batches we need to pad them to the same length. `DataCollatorForTokenClassification` does this for us, and it pads the labels with `-100` as well:

In [ ]:
collator = DataCollatorForTokenClassification(tokenizer)

train_loader = DataLoader(train_tok, batch_size=16, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_tok, batch_size=64, collate_fn=collator)
test_loader = DataLoader(test_tok, batch_size=64, collate_fn=collator)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}

## Loading the Pre-trained Model

`AutoModelForTokenClassification` loads the pre-trained DistilBERT and adds a new linear layer on top of it, which maps the output vector of each token to the 9 classes. You will see a warning that some weights were newly initialized: this is expected, because the classification layer has not been trained yet.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name, num_labels=len(label_names), id2label=id2label, label2id=label2id
).to(device)

## Training

Because the model is already pre-trained, we use a small learning rate (`5e-5`) so that we do not destroy what it has already learned. We also gradually decrease the learning rate during training using a scheduler. The training loop is the same as for any other PyTorch model: when we pass `labels` to the model, it computes the cross-entropy loss for us (ignoring the `-100` positions).

> Training for 3 epochs on 2,000 sentences takes a few minutes on a CPU. If you want faster experiments, reduce the number of epochs or the size of the training subset.

In [ ]:
epochs = 3
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
total_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        if (step + 1) % 25 == 0:
            print(f'Epoch {epoch + 1}, step {step + 1}/{len(train_loader)}: average loss = {total_loss / (step + 1):.4f}')
    print(f'Epoch {epoch + 1} done, average loss = {total_loss / len(train_loader):.4f}')

## Evaluation

To evaluate the model, we run it on the validation set and take the class with the highest score for each token. Then we need to convert the predictions back to **word-level** tag sequences. Since the first sub-word token of each word is the only one with a label different from `-100`, we can simply skip all positions where the label is `-100`:

In [ ]:
def predict_tags(data_loader):
    model.eval()
    true_tags, pred_tags = [], []
    with torch.no_grad():
        for batch in data_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            predictions = logits.argmax(dim=-1)
            for pred_row, label_row in zip(predictions.tolist(), batch['labels'].tolist()):
                true_tags.append([id2label[l] for l in label_row if l != -100])
                pred_tags.append([id2label[p] for p, l in zip(pred_row, label_row) if l != -100])
    return true_tags, pred_tags

val_true, val_pred = predict_tags(val_loader)

First, let's compute **token accuracy** - the percentage of words with a correct tag - and compare it to a "model" that always predicts `O`:

In [ ]:
flat_true = [t for seq in val_true for t in seq]
flat_pred = [p for seq in val_pred for p in seq]

accuracy = np.mean([t == p for t, p in zip(flat_true, flat_pred)])
all_o_accuracy = np.mean([t == 'O' for t in flat_true])

print(f'Token accuracy of our model:          {accuracy:.1%}')
print(f'Token accuracy of "always predict O": {all_o_accuracy:.1%}')

The accuracy of our model is high, but the useless "always `O`" baseline also gets a high score, so this number alone does not tell us much. Let's use **entity-level** metrics from the `seqeval` library instead. An entity is counted as correct only if its type and **all of its boundaries** are predicted correctly:

In [ ]:
print(f"Entity-level precision: {precision_score(val_true, val_pred):.3f}")
print(f"Entity-level recall:    {recall_score(val_true, val_pred):.3f}")
print(f"Entity-level F1:        {f1_score(val_true, val_pred):.3f}")
print()
print(classification_report(val_true, val_pred))

The "always `O`" baseline would have an F1 score of exactly 0, so the entity-level metrics clearly show that our model has learned something useful. Notice also that the F1 score is lower than the token accuracy, because getting just one word of a multi-word name wrong makes the whole entity wrong.

> ✅ Which entity types are the easiest and which are the hardest for the model? Why do you think this is the case? (Hint: `MISC` is a mixture of many different things, such as nationalities and events.)

Finally, we check the model on the **test** set, which we have not looked at during training or when choosing the settings. The test set is the number you would report as the final quality of your model:

In [ ]:
test_true, test_pred = predict_tags(test_loader)
print(f"Test F1: {f1_score(test_true, test_pred):.3f}")

## Trying the Model on Our Own Sentences

Now the fun part: let's see what the model finds in sentences that it has never seen. We need to repeat the same steps as during training: split the text into words, tokenize it, run the model, and then read the prediction of the **first sub-word token of each word** (remember that these are the only positions that the model was trained on).

We will split the text into words with a simple regular expression that separates punctuation marks from words. Then we will merge the `B-`/`I-` tags back into entities, as described in the lesson:

In [ ]:
import re

def tags_to_entities(words, tags):
    entities, current = [], None
    for word, tag in zip(words, tags):
        if tag.startswith('B-'):                          # a new entity starts
            current = [tag[2:], [word]]
            entities.append(current)
        elif tag.startswith('I-') and current and current[0] == tag[2:]:
            current[1].append(word)                       # continue the current entity
        else:                                             # "O" closes the current entity
            current = None
    return [(label, ' '.join(ws)) for label, ws in entities]

def find_entities(sentence):
    words = re.findall(r"\w+|[^\w\s]", sentence)
    encoding = tokenizer(words, is_split_into_words=True, return_tensors='pt').to(device)
    model.eval()
    with torch.no_grad():
        predictions = model(**encoding).logits.argmax(dim=-1)[0].tolist()
    # keep the prediction of the first sub-word token of each word
    tags, previous_word = [], None
    for token_index, word_id in enumerate(encoding.word_ids()):
        if word_id is not None and word_id != previous_word:
            tags.append(id2label[predictions[token_index]])
        previous_word = word_id
    return words, tags

sentences = [
    'Satya Nadella is the CEO of Microsoft, which is headquartered in Redmond.',
    'Angela Merkel met the French president in Berlin on Monday.',
    'The Eiffel Tower in Paris attracts millions of visitors every year.',
]

for sentence in sentences:
    words, tags = find_entities(sentence)
    print(sentence)
    print('   Tags:    ', ' '.join(f'{w}/{t}' for w, t in zip(words, tags) if t != 'O'))
    print('   Entities:', tags_to_entities(words, tags))
    print()

Try your own sentences, too! You will probably notice some mistakes, especially for names that look different from the news articles in CoNLL-2003, or for text that is not written in a news style (for example social media posts or medical text). This is expected: the model only saw 2,000 sentences, and the training data is quite specific.

> ✅ What could you do to improve the model? Think about using more training data, training for more epochs, and using a larger or a domain-specific pre-trained model.

## Challenge

* Increase the size of the training subset and the number of epochs. How does the F1 score on the validation set change?
* Try a different pre-trained model, such as `bert-base-cased`, by changing `model_name`.
* Instead of using only the first sub-word token for each word, try labeling all the pieces of a word (using the `I-` tag for continuation pieces). Does it make any difference?
* The [assignment](lab/README.md) of this lesson asks you to train a NER model for medical terms. The same steps (BIO tags, sub-word alignment, fine-tuning, entity-level evaluation) apply there, too.